# Practical 4: POS Tagging using Hidden Markov Model (HMM) + Viterbi

**Idea:** tags are the hidden states, words are the observations.
We learn two things from a tagged corpus:
- **Transition probability** P(tag | previous tag)
- **Emission probability** P(word | tag)

Then Viterbi finds the best tag sequence for a new sentence.

In [1]:
# 1. Imports and setup
import random
import math
from collections import Counter, defaultdict

import nltk
nltk.download('treebank', quiet=True)
nltk.download('universal_tagset', quiet=True)
from nltk.corpus import treebank

random.seed(42)
print("Setup done")

Setup done


## 2. Load the tagged corpus and split it into train / test

In [2]:
# Penn Treebank sample from NLTK, using the simple Universal tagset (12 tags)
sents = list(treebank.tagged_sents(tagset='universal'))
random.shuffle(sents)

split = int(0.8 * len(sents))
train_sents = sents[:split]
test_sents  = sents[split:]

print("Total sentences :", len(sents))
print("Train sentences :", len(train_sents))
print("Test sentences  :", len(test_sents))
print("Example         :", train_sents[0][:6])

Total sentences : 3914
Train sentences : 3131
Test sentences  : 783
Example         : [('The', 'DET'), ('company', 'NOUN'), ('noted', 'VERB'), ('that', 'ADP'), ('it', 'PRON'), ('has', 'VERB')]


## 3. Estimate HMM parameters (counting)


In [3]:
tag_count  = Counter()
transition = defaultdict(Counter)   # transition[prev_tag][tag]
emission   = defaultdict(Counter)   # emission[tag][word]
word_count = Counter()

for sent in train_sents:
    prev = '<START>'
    for word, tag in sent:
        w = word.lower()
        tag_count[tag] += 1
        transition[prev][tag] += 1
        emission[tag][w] += 1
        word_count[w] += 1
        prev = tag
    transition[prev]['<END>'] += 1

TAGS  = sorted(tag_count)
VOCAB = set(word_count)

print("Tags      :", TAGS)
print("Vocab size:", len(VOCAB))
print("Most common tags:", tag_count.most_common(5))

Tags      : ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
Vocab size: 10108
Most common tags: [('NOUN', 23202), ('VERB', 10888), ('.', 9455), ('ADP', 7924), ('DET', 7026)]


## 4. Unknown-word handling
A word never seen in training would get probability 0 and break the whole path.
Two strategies are compared:

1. **Basic (Laplace)**: add-1 smoothing on emissions. Unknown words get a tiny equal-ish probability.
2. **Improved (word signature)**: an unknown word is replaced by a *signature* based on its shape
   (capital letter, number, suffix like -ing, -ed, -ly ...). We learn P(signature | tag) from
   words that occur only once in training (they behave like unknown words).

In [4]:
def signature(word):
    # Map a word to a rough 'shape' class. Used for unseen words.
    w = word.lower()
    if any(ch.isdigit() for ch in w):
        return '<UNK-NUM>'
    if word[0].isupper():
        return '<UNK-CAP>'
    if '-' in w:
        return '<UNK-HYPHEN>'
    for suf, name in [('ing','ING'), ('ed','ED'), ('ly','LY'), ('tion','TION'), ('sion','TION'),
                      ('ment','MENT'), ('ness','NESS'), ('able','ABLE'), ('ible','ABLE'),
                      ('ous','OUS'), ('ive','IVE'), ('ful','FUL'), ('al','AL'), ('ic','IC'),
                      ('ity','ITY'), ('er','ER'), ('s','S')]:
        if w.endswith(suf):
            return '<UNK-' + name + '>'
    return '<UNK>'

# Learn signature emissions from words seen only once (hapax words)
sig_emission = defaultdict(Counter)
for sent in train_sents:
    for word, tag in sent:
        if word_count[word.lower()] == 1:
            sig_emission[tag][signature(word)] += 1

SIGS = set(s for t in sig_emission for s in sig_emission[t]) | {'<UNK>'}
print("Number of signatures learned:", len(SIGS))

Number of signatures learned: 19


## 5. Probability functions (log scale)
Here I used **log probabilities** so many small numbers do not underflow.

In [5]:
K_TRANS = 0.01       # additive smoothing for transitions
NT = len(TAGS) + 1   # possible next states (all tags + <END>)

def log_trans(prev, tag):
    total = sum(transition[prev].values())
    return math.log((transition[prev][tag] + K_TRANS) / (total + K_TRANS * NT))

def log_emit(tag, word, mode):
    w = word.lower()
    if w in VOCAB:
        # known word: mild smoothing so a rare (tag, word) pair is never exactly 0
        return math.log((emission[tag][w] + 1e-3) / (tag_count[tag] + 1e-3 * (len(VOCAB) + 1)))
    # ---- unknown word ----
    if mode == 'laplace':
        return math.log(1 / (tag_count[tag] + len(VOCAB) + 1))
    sig = signature(word)
    if sig not in SIGS:
        sig = '<UNK>'
    n_tag = sum(sig_emission[tag].values())
    return math.log((sig_emission[tag][sig] + 0.1) / (n_tag + 0.1 * len(SIGS)))

## 6. Viterbi algorithm
For each position and tag we keep the best score of any path ending in that tag,
plus a backpointer. At the end we backtrack to get the best tag sequence.

In [6]:
def viterbi(words, mode='signature', return_table=False):
    n = len(words)
    best = [{} for _ in range(n)]   # best[i][tag] = best log score
    back = [{} for _ in range(n)]   # back[i][tag] = previous tag

    # first word
    for t in TAGS:
        best[0][t] = log_trans('<START>', t) + log_emit(t, words[0], mode)
        back[0][t] = None

    # remaining words
    for i in range(1, n):
        for t in TAGS:
            e = log_emit(t, words[i], mode)
            prev_best = max(TAGS, key=lambda p: best[i-1][p] + log_trans(p, t))
            best[i][t] = best[i-1][prev_best] + log_trans(prev_best, t) + e
            back[i][t] = prev_best

    # end transition
    last = max(TAGS, key=lambda t: best[n-1][t] + log_trans(t, '<END>'))
    final_score = best[n-1][last] + log_trans(last, '<END>')

    # backtrack
    path = [last]
    for i in range(n - 1, 0, -1):
        path.append(back[i][path[-1]])
    path.reverse()

    return (path, final_score) if return_table else path

print(viterbi("the dog runs".split()))

['DET', 'NOUN', '.']


### Quick check: is Viterbi correct?
For a short sentence i try **every** possible tag sequence (brute force) and confirm that Viterbi finds the same best one.

In [7]:
from itertools import product

def sequence_score(words, tags, mode='signature'):
    s = log_trans('<START>', tags[0])
    for i, (w, t) in enumerate(zip(words, tags)):
        s += log_emit(t, w, mode)
        if i > 0:
            s += log_trans(tags[i-1], t)
    return s + log_trans(tags[-1], '<END>')

test_words = "the market rose sharply".split()
brute = max(product(TAGS, repeat=len(test_words)), key=lambda seq: sequence_score(test_words, seq))
vit, vit_score = viterbi(test_words, return_table=True)

print("Brute force:", list(brute))
print("Viterbi    :", vit)
print("Same result :", list(brute) == vit)

Brute force: ['DET', 'NOUN', 'VERB', 'ADV']
Viterbi    : ['DET', 'NOUN', 'VERB', 'ADV']
Same result : True


## 7. Tag unseen sentences (Question 1)

In [8]:
new_sentences = [
    "The company reported strong profits last quarter .",
    "She quickly opened the old wooden door .",
    "Investors are watching the market closely .",
    "The government will announce new rules on Monday .",
]
for s in new_sentences:
    words = s.split()
    tags = viterbi(words)
    print(" ".join(f"{w}/{t}" for w, t in zip(words, tags)))
    print()

The/DET company/NOUN reported/VERB strong/ADJ profits/NOUN last/ADJ quarter/NOUN ./.

She/PRON quickly/ADV opened/VERB the/DET old/ADJ wooden/NOUN door/NOUN ./.

Investors/NOUN are/VERB watching/VERB the/DET market/NOUN closely/ADV ./.

The/DET government/NOUN will/VERB announce/VERB new/ADJ rules/NOUN on/ADP Monday/NOUN ./.



## 8. Accuracy: before vs after unknown-word handling (Question 2)

In [9]:
def evaluate(data, mode):
    correct = total = 0
    unk_correct = unk_total = 0
    results = []
    for sent in data:
        words = [w for w, _ in sent]
        gold  = [t for _, t in sent]
        pred  = viterbi(words, mode)
        results.append((words, gold, pred))
        for w, g, p in zip(words, gold, pred):
            total += 1
            correct += (g == p)
            if w.lower() not in VOCAB:
                unk_total += 1
                unk_correct += (g == p)
    return correct / total, unk_correct / max(unk_total, 1), unk_total / total, results

acc_b, unk_b, unk_rate, res_b = evaluate(test_sents, 'laplace')
acc_s, unk_s, _,        res_s = evaluate(test_sents, 'signature')

print(f"Unknown words in test set: {unk_rate*100:.1f}% of tokens\n")
print(f"{'Method':<28}{'Overall acc':>12}{'Unknown-word acc':>20}")
print(f"{'Basic (Laplace)':<28}{acc_b*100:>11.2f}%{unk_b*100:>19.2f}%")
print(f"{'Improved (signature)':<28}{acc_s*100:>11.2f}%{unk_s*100:>19.2f}%")

Unknown words in test set: 7.0% of tokens

Method                       Overall acc    Unknown-word acc
Basic (Laplace)                   93.06%              47.30%
Improved (signature)              95.17%              77.19%


## 9. Aligned output for 5 test sentences (Question 3)

In [10]:
# pick 5 different test sentences of medium length
shown, seen = [], set()
for words, gold, pred in res_s:
    key = ' '.join(words)
    if 8 <= len(words) <= 18 and key not in seen:
        seen.add(key)
        shown.append((words, gold, pred))
    if len(shown) == 5:
        break

for words, gold, pred in shown:
    print(f"{'WORD':<22}{'ACTUAL':<9}{'PREDICTED':<10}")
    for w, g, p in zip(words, gold, pred):
        flag = '' if g == p else '  <-- wrong'
        print(f"{w:<22}{g:<9}{p:<10}{flag}")
    print('-' * 45)

WORD                  ACTUAL   PREDICTED 
Drexel                NOUN     NOUN      
remains               VERB     VERB      
confident             ADJ      NOUN        <-- wrong
of                    ADP      ADP       
its                   PRON     PRON      
future                ADJ      ADJ       
creditworthiness      NOUN     NOUN      
.                     .        .         
---------------------------------------------
WORD                  ACTUAL   PREDICTED 
INQUIRY               NOUN     NOUN      
CLEARS                VERB     NOUN        <-- wrong
TEXAS                 NOUN     NOUN      
JUDGE                 NOUN     NOUN      
of                    ADP      ADP       
bias                  NOUN     NOUN      
in                    ADP      ADP       
comments              NOUN     NOUN      
on                    ADP      ADP       
homosexual            ADJ      NOUN        <-- wrong
murder                NOUN     DET         <-- wrong
victims               NOUN  

## 10. Error analysis: most frequent confusions

In [11]:
confusions = Counter()
bad_words = Counter()
for (words, gold, pred) in res_s:
    for w, g, p in zip(words, gold, pred):
        if g != p:
            confusions[(g, p)] += 1
            bad_words[(w.lower(), g, p)] += 1

print("Top 8 confusions (actual -> predicted):")
for (g, p), c in confusions.most_common(8):
    print(f"  {g:>5} -> {p:<5} : {c}")

print("\nTop 10 words tagged wrongly:")
for (w, g, p), c in bad_words.most_common(10):
    print(f"  {w:<12} actual={g:<5} predicted={p:<5} x{c}")

Top 8 confusions (actual -> predicted):
   NOUN -> ADJ   : 147
   VERB -> NOUN  : 120
    ADJ -> NOUN  : 92
   NOUN -> VERB  : 84
    DET -> ADP   : 43
    ADJ -> ADV   : 35
   VERB -> ADP   : 31
   VERB -> ADJ   : 29

Top 10 words tagged wrongly:
  that         actual=DET   predicted=ADP   x43
  new          actual=NOUN  predicted=ADJ   x31
  that         actual=ADP   predicted=DET   x25
  there        actual=DET   predicted=ADV   x13
  more         actual=ADJ   predicted=ADV   x10
  more         actual=ADV   predicted=ADJ   x10
  big          actual=NOUN  predicted=ADJ   x8
  's           actual=VERB  predicted=PRT   x8
  american     actual=NOUN  predicted=ADJ   x8
  up           actual=PRT   predicted=ADV   x8


## 11. Observations (fill in using your own output)
- Two error patterns to discuss, based on the confusion table above:
  1. **Ambiguous words** such as *that* (DET vs ADP), *there*, *more* and *up* belong to several tags. The HMM only sees the previous tag, so it often picks the commonest reading.
  2. **NOUN vs ADJ / VERB vs NOUN confusion**. Words like *new* or *american* are labelled NOUN inside proper names (New York), and many words (e.g. *plan*, *offer*) can be both noun and verb.
- Compare the Laplace and signature results above: the signature strategy mainly helps on unknown words.

## 12. Conclusion
Write 3 to 4 lines: HMM with Viterbi gives good accuracy using only tag-bigram context, unknown-word handling improved accuracy, and the remaining errors come from ambiguous words and limited context.